## <U>Etapa 1</U>: Introduccion e importacion de bibliotecas

## Objetivo:
### Construir un sistema de clasificacion que permita relacionar las carreras de nivel superior disponibles en Jujuy con distintas áreas de conocimiento.
### Este mapeo servirá como base para el futuro sistema de orientacion vocacional, permitiendo relacionar los intereses y las fortalezas academicas de los estudiantes con posibles alternativas educativas.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata

print("Biblitecas importadas correctamente")

Biblitecas importadas correctamente


### Importando el archivo

In [2]:
ruta_catalogo = "../Datos/Procesados/catalogo_carreras_jujuy_limpio.csv"

try:
    df_carreras = pd.read_csv(ruta_catalogo)
    
    print("Catalogo cargado correctamente")
    print(f"Registros:{len(df_carreras)}")

except FileNotFoundError:
    print("Error: no se encontro el archivo")

Catalogo cargado correctamente
Registros:463


### Primera exploracion

In [3]:
print("Dimensiones del catalogo: ")
print(df_carreras.shape)

print("\nColumnas disponibles: ")
print(df_carreras.columns.tolist())

print("\nPrimeros 15 registros: ")
display(df_carreras.head(15))

print("\nInformacion general: ")
df_carreras.info()

Dimensiones del catalogo: 
(463, 7)

Columnas disponibles: 
['Carrera', 'Categoria', 'Localidad', 'Institucion', 'Archivo_origen', 'Localidad_normalizada', 'Institucion_normalizada']

Primeros 15 registros: 


,Carrera,Categoria,Localidad,Institucion,Archivo_origen,Localidad_normalizada,Institucion_normalizada
0,Profesorado de Educación Tecnológica,Tecnologia,Abra Pampa,Instituto de Educación Superior N° 1,tecnologia_jujuy.csv,Abra Pampa,Instituto de Educación Superior N° 1
1,Tecnicatura superior en ciencia de datos e int...,Tecnologia,Abra Pampa,Instituto de Educación Superior N° 1,tecnologia_jujuy.csv,Abra Pampa,Instituto de Educación Superior N° 1
2,Profesorado de Educación Secundaria en Informá...,Tecnologia,Humahuaca,Instituto de Educación Superior N° 2,tecnologia_jujuy.csv,Humahuaca,Instituto de Educación Superior N° 2
3,Tecnicatura superior en desarrollo de software,Tecnologia,Humahuaca,Instituto de Educación Superior N° 2,tecnologia_jujuy.csv,Humahuaca,Instituto de Educación Superior N° 2
4,Tecnicatura superior en mantenimiento industrial,Tecnologia,Maimara,Instituto de Educación Superior N° 2,tecnologia_jujuy.csv,Maimará,Instituto de Educación Superior N° 2
5,Tecnicatura superior en mantenimiento industrial,Tecnologia,Susques,Instituto de Educación Superior N° 3,tecnologia_jujuy.csv,Susques,"Instituto de Educación Superior N° 3 ""Escuela ..."
6,Profesorado de Educación Secundaria en Tecnolo...,Tecnologia,San Salvador de Jujuy,Instituto de Educación Superior N° 5,tecnologia_jujuy.csv,San Salvador de Jujuy,Instituto de Educación Superior N° 5
7,Profesorado de Educación Tecnológica,Tecnologia,San Salvador de Jujuy,Instituto de Educación Superior N° 5,tecnologia_jujuy.csv,San Salvador de Jujuy,Instituto de Educación Superior N° 5
8,Tecnicatura superior en soporte de infraestruc...,Tecnologia,San Salvador de Jujuy,Instituto de Educación Superior N° 5,tecnologia_jujuy.csv,San Salvador de Jujuy,Instituto de Educación Superior N° 5
9,Profesorado de Educación Secundaria en Informá...,Tecnologia,Perico,Instituto de Educación Superior N° 6,tecnologia_jujuy.csv,Perico,Instituto de Educación Superior N° 6



Informacion general: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 463 entries, 0 to 462
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Carrera                  463 non-null    object
 1   Categoria                463 non-null    object
 2   Localidad                463 non-null    object
 3   Institucion              463 non-null    object
 4   Archivo_origen           463 non-null    object
 5   Localidad_normalizada    463 non-null    object
 6   Institucion_normalizada  463 non-null    object
dtypes: object(7)
memory usage: 25.4+ KB


## <U>Etapa 2</U>: Definicion de las areas de conocimiento
### Para organizar la oferta academica se establecera una clasificacion tematica propia, agrupando las carreras segun su campo principal de estudio.
### Se utilizaran ocho areas generales. Esta clasificacion permitira relacionar posteriormente las carreras con las materias escolares, los intereses y las fortalezas academicas de los estudiantes.
### **Una carrera podra tener un area principal y, cuando corresponda, un area secundaria**

In [4]:
areas_conocimiento = {
    "AC01":"Ciencias Exactas y Naturales",
    "AC02":"Salud y Ciencias de la Vida",
    "AC03":"Tecnologia e Informatica",
    "AC04":"Ciencias Sociales y Humanas",
    "AC05":"Educacion y Formacion Docente",
    "AC06":"Economia, Administracion y Negocios",
    "AC07":"Arte, Diseño y Comunicacion",
    "AC08":"Ambiente, Produccion y Ciencias Agropecuarias"
}
print("Diccionario creado correctamente")

Diccionario creado correctamente


In [5]:
df_areas = pd.DataFrame(
    list(areas_conocimiento.items()),
    columns = ("Codigo_area","Area_conocimiento")
)
display(df_areas)

,Codigo_area,Area_conocimiento
0,AC01,Ciencias Exactas y Naturales
1,AC02,Salud y Ciencias de la Vida
2,AC03,Tecnologia e Informatica
3,AC04,Ciencias Sociales y Humanas
4,AC05,Educacion y Formacion Docente
5,AC06,"Economia, Administracion y Negocios"
6,AC07,"Arte, Diseño y Comunicacion"
7,AC08,"Ambiente, Produccion y Ciencias Agropecuarias"


## <U>Etapa 3</U>: Analisis de las categorias originales
### Antes de clasificar las carreras, se analizaran las categorias de origen del catalogo unificado

In [6]:
columnas_texto = df_carreras.select_dtypes(include=["object","string"]).columns
print("columnas categoricas disponibles: ")
for columna in columnas_texto:
    print(f"-{columna}")

columnas categoricas disponibles: 
-Carrera
-Categoria
-Localidad
-Institucion
-Archivo_origen
-Localidad_normalizada
-Institucion_normalizada


In [7]:
for columna in columnas_texto:
    print(f"\n{'=' * 60}")
    print(f"COLUMNA: {columna}")
    print(f"Valores unicos: {df_carreras[columna].nunique()}")

    print("\nFrecuencias: ")
    display(
        df_carreras[columna].value_counts(dropna=False).head(30)
    )


COLUMNA: Carrera
Valores unicos: 264

Frecuencias: 


Carrera
Tecnicatura superior en desarrollo de software                                        11
Enfermería                                                                            11
Tecnicatura superior en higiene y seguridad en el trabajo                              9
Tecnicatura superior en agente sanitario y promotor de la salud                        9
Prof. de Educ. Sec. en Lengua y Literatura                                             9
Tecnicatura superior en ciencia de datos e inteligencia artificial                     8
Tecnicatura superior en Cocinas Regionales y Cultura Alimentaria                       7
Profesorado de Inglés                                                                  7
Tecnicatura superior en hemoterapia                                                    7
Tecnicatura superior en turismo                                                        6
Profesorado de Educación Secundaria en Ciencia Política                                6
Tecnicatura s


COLUMNA: Categoria
Valores unicos: 14

Frecuencias: 


Categoria
Tecnicaturas                        154
Profesorados                         91
Salud Y Psicologia                   40
Ciencias Sociales                    35
Negocio Y Administracion             29
Tecnologia                           22
Ciencias Exactas                     17
Arte Y Diseño                        15
Medio Ambiente Y Sustentabilidad     15
Turismo Hoteleria Y Cocina           13
Derecho                              12
Idiomas                              10
Rural                                 5
Deportes                              5
Name: count, dtype: int64


COLUMNA: Localidad
Valores unicos: 31

Frecuencias: 


Localidad
San Salvador de Jujuy                         249
San Pedro                                      41
Perico                                         20
Monterrico                                     19
Humahuaca                                      18
Abra Pampa                                     17
Libertador Gral. San Martín                    16
La Quiaca                                      11
Tilcara                                        11
Libertador                                     10
El Carmen                                      10
Fraile Pintado                                  6
Palpalá - Rio Blanco                            5
Palpala                                         4
San Pedro de Jujuy                              3
La Mendieta                                     3
Rodeito                                         2
Palpalá                                         2
Yuto                                            2
Tumbaya                                 

... (4 outputs left out)


COLUMNA: Localidad_normalizada
Valores unicos: 28

Frecuencias: 


Localidad_normalizada
San Salvador de Jujuy                         250
San Pedro                                      41
Perico                                         20
Monterrico                                     19
Humahuaca                                      18
Abra Pampa                                     17
Libertador General San Martín                  17
La Quiaca                                      11
Tilcara                                        11
Libertador                                     10
El Carmen                                      10
Palpalá                                         6
Fraile Pintado                                  6
Palpalá - Rio Blanco                            5
San Pedro de Jujuy                              3
La Mendieta                                     3
Rodeito                                         2
Yuto                                            2
Maimará                                         2
Tumbaya                     


COLUMNA: Institucion_normalizada
Valores unicos: 40

Frecuencias: 


Institucion_normalizada
Universidad Siglo 21                                                          51
Instituto de Educación Superior N° 9 "Juana Azurduy"                          44
Instituto de Educación Superior N° 6                                          33
Instituto de Educación Superior "Nuevo Horizonte"                             31
Instituto de Educación Superior N° 7 "Populorum Progressio In.Te.La."         28
Instituto de Educación Superior N° 10                                         27
Instituto de Educación Superior N° 1                                          26
Instituto de Educación Superior N° 2                                          26
Instituto de Educación Superior N° 11                                         26
Instituto de Educación Superior N° 4                                          24
Instituto de Educación Superior N° 5 "José Eugenio Tello"                     20
Instituto de Educación Superior N° 5                                          18
Inst

## <U>Etapa 3</U>: Estrategia de clasificacion de las categorias.
### Se establecera una correspondencia inicial entre las categorias originales de Expo educativa y las areas de conocimiento del proyecto.
### Debido a que algunas categorias agrupan carreras de distintas disciplinas, no todas podran asignarse directamente a una unica area. En estos casos, la clasificacion se realizara considerando el nombre y campo academico de cada carrera.
### **Esta estrategia permitira conservar la informacion original y evitar asignaciones incorrectas**

### Categorias que pueden orientar una asignacion directa

In [8]:
mapeo_directo = {
    "Tecnologia":"AC03",
    "Ciencias Exactas":"AC01",
    "Arte y Diseño":"AC07",
    "Ciencias Sociales":"AC04",
    "Negocio y Administracion":"AC06",
    "Derecho":"AC04",
    "Rural":"AC08"
}
print("Creado correctamente")

Creado correctamente


### Categorias que requieren analizar cada carrera

In [9]:
categoria_revision = {
    "Tecnicaturas",
    "Profesorados",
    "Salud y Psicologia",
    "Medio Ambiente y Sustentabilidad",
    "Turismo Hoteleria y Cocina",
    "Idiomas",
    "Deportes",
}
print("Creado correctamente")

Creado correctamente


In [10]:
print("Categorias de asignacion directa: ")
for categoria, codigo in mapeo_directo.items():
    print(f"{categoria}->{codigo}: {areas_conocimiento[codigo]}")

print("\nCategorias que requieren revision individual: ")
for categoria in categoria_revision:
    print("-",categoria)

Categorias de asignacion directa: 
Tecnologia->AC03: Tecnologia e Informatica
Ciencias Exactas->AC01: Ciencias Exactas y Naturales
Arte y Diseño->AC07: Arte, Diseño y Comunicacion
Ciencias Sociales->AC04: Ciencias Sociales y Humanas
Negocio y Administracion->AC06: Economia, Administracion y Negocios
Derecho->AC04: Ciencias Sociales y Humanas
Rural->AC08: Ambiente, Produccion y Ciencias Agropecuarias

Categorias que requieren revision individual: 
- Medio Ambiente y Sustentabilidad
- Salud y Psicologia
- Deportes
- Tecnicaturas
- Turismo Hoteleria y Cocina
- Profesorados
- Idiomas


### Comprobacion

In [11]:
print("Categorias originales: ",df_carreras["Categoria"].nunique())

print("Categorias con asignacion directa: ",len(mapeo_directo))

print("Categorias para revision: ",len(categoria_revision))

print("\nTotal: ",len(mapeo_directo)+len(categoria_revision))

Categorias originales:  14
Categorias con asignacion directa:  7
Categorias para revision:  7

Total:  14


### Extraer las carreras para revision

In [29]:
df_revision = df_carreras[
    df_carreras["Categoria"].str.casefold().isin({c.casefold() for c in categoria_revision})
    ].copy()

print("Registros que requieren revision:", len(df_revision))
print("Carreras distintas:", df_revision["Carrera"].nunique())

display(
    df_revision.groupby("Categoria").agg(Registros=("Carrera", "size"),Carreras_distintas=("Carrera", "nunique"))
)

Registros que requieren revision: 328
Carreras distintas: 184


,Registros,Carreras_distintas
Categoria,,
Deportes,5,3
Idiomas,10,4
Medio Ambiente Y Sustentabilidad,15,10
Profesorados,91,72
Salud Y Psicologia,40,25
Tecnicaturas,154,91
Turismo Hoteleria Y Cocina,13,9


### Obtener el listado de carreras

In [30]:
carreras_division = (
    df_revision[["Categoria","Carrera"]].drop_duplicates().sort_values(["Categoria","Carrera"]).reset_index(drop=True)
)

display(carreras_division)

,Categoria,Carrera
0,Deportes,Profesorado de Educación Física
1,Deportes,Profesorado de Educación Física - Orientación ...
2,Deportes,Tecnicatura superior en entrenamiento deportivo
3,Idiomas,Profesorado de Educación Primaria - Orientació...
4,Idiomas,Profesorado de Francés
...,...,...
209,Turismo Hoteleria Y Cocina,Tecnicatura superior en guía de turismo
210,Turismo Hoteleria Y Cocina,Tecnicatura superior en guía de turismo comuni...
211,Turismo Hoteleria Y Cocina,Tecnicatura superior en hotelería
212,Turismo Hoteleria Y Cocina,Tecnicatura superior en tecnología de los alim...


### Contar las carreras por categoria 

In [31]:
resumen_revision = (
    carreras_division.groupby("Categoria").agg(Cantidad_carreras = ("Carrera","count")).reset_index()
)

display(resumen_revision)

,Categoria,Cantidad_carreras
0,Deportes,3
1,Idiomas,4
2,Medio Ambiente Y Sustentabilidad,10
3,Profesorados,72
4,Salud Y Psicologia,25
5,Tecnicaturas,91
6,Turismo Hoteleria Y Cocina,9


### Comprobar categorias faltantes

In [32]:
print("Todas las categorias del catalogo: \n")
display(df_carreras["Categoria"].value_counts(dropna=False))

print("\nCategorias configuradas para revision: ")
print(categoria_revision)

Todas las categorias del catalogo: 



Categoria
Tecnicaturas                        154
Profesorados                         91
Salud Y Psicologia                   40
Ciencias Sociales                    35
Negocio Y Administracion             29
Tecnologia                           22
Ciencias Exactas                     17
Arte Y Diseño                        15
Medio Ambiente Y Sustentabilidad     15
Turismo Hoteleria Y Cocina           13
Derecho                              12
Idiomas                              10
Rural                                 5
Deportes                              5
Name: count, dtype: int64


Categorias configuradas para revision: 
{'Medio Ambiente y Sustentabilidad', 'Salud y Psicologia', 'Deportes', 'Tecnicaturas', 'Turismo Hoteleria y Cocina', 'Profesorados', 'Idiomas'}


In [33]:
print("Categorias que realmente ingresaron a revision: ")
print(sorted(df_revision["Categoria"].dropna().unique()))

print("\nCategorias que no ingresaron: ")

categorias_en_catalogo = set(df_carreras["Categoria"].dropna().unique())

categorias_en_revision = set(df_revision["Categoria"].dropna().unique())

print(categorias_en_catalogo - categorias_en_revision)

Categorias que realmente ingresaron a revision: 
['Deportes', 'Idiomas', 'Medio Ambiente Y Sustentabilidad', 'Profesorados', 'Salud Y Psicologia', 'Tecnicaturas', 'Turismo Hoteleria Y Cocina']

Categorias que no ingresaron: 
{'Ciencias Exactas', 'Ciencias Sociales', 'Negocio Y Administracion', 'Derecho', 'Arte Y Diseño', 'Rural', 'Tecnologia'}


## <U>Etapa 4</U>: Construccion de la tabla base de mapeos
### Se creara una tabla de carreras unicas a partir del catalogo depurado. Se conservara su categoria original para utilizarla como referencia durante la asignacion de areas de conocimiento.
### Las distintas ofertas de una misma carrera se antendran en el catalogo original, evitando perder informacion institucional y territorial

In [34]:
df_mapeo = (
    df_carreras[["Carrera","Categoria"]].drop_duplicates().sort_values("Carrera").reset_index(drop=True)
)
print("Carreras en la tabla de mapeo: ",len(df_mapeo))

display(df_mapeo.head(20))

Carreras en la tabla de mapeo:  312


,Carrera,Categoria
0,Abogacía,Derecho
1,Abogacía - Modalidad Online,Ciencias Sociales
2,Actuario,Negocio Y Administracion
3,Contador Publico,Negocio Y Administracion
4,Enfermeria,Tecnicaturas
5,Enfermería,Tecnicaturas
6,Enfermería,Salud Y Psicologia
7,Enfermería Universitaria,Salud Y Psicologia
8,Ingeniería Agronómica,Rural
9,Lic en Fonoaudiología,Salud Y Psicologia


### incorporar las columnas de clasificacion

In [35]:
df_mapeo["Codigo_area"] = ""
df_mapeo["Area_principal"] = ""
df_mapeo["Subarea"] = ""
df_mapeo["Criterio_asignacion"] = ""

display(df_mapeo.head(10))

,Carrera,Categoria,Codigo_area,Area_principal,Subarea,Criterio_asignacion
0,Abogacía,Derecho,,,,
1,Abogacía - Modalidad Online,Ciencias Sociales,,,,
2,Actuario,Negocio Y Administracion,,,,
3,Contador Publico,Negocio Y Administracion,,,,
4,Enfermeria,Tecnicaturas,,,,
5,Enfermería,Tecnicaturas,,,,
6,Enfermería,Salud Y Psicologia,,,,
7,Enfermería Universitaria,Salud Y Psicologia,,,,
8,Ingeniería Agronómica,Rural,,,,
9,Lic en Fonoaudiología,Salud Y Psicologia,,,,


In [36]:
df_mapeo.shape

(312, 6)

### Comprobar carreras unicas

In [37]:
print("Filas del mapeo: ",len(df_mapeo))

print("Nombres de carreras unicos: ",df_mapeo["Carrera"].nunique())

print("Categorias originales: ",df_mapeo["Categoria"].nunique())

Filas del mapeo:  312
Nombres de carreras unicos:  264
Categorias originales:  14


## <U>Etapa 5</U>: Clasificiacion de carreas por area de conocimiento
### Se realizara la asignacion de las carreras a las areas de conocimiento definidas anteriormente.
### La clasificacion considerara el nombre de cada carrera y su campo academico. Se conservaran las categorias originales como referencia y se evitaran asignaciones automaticas cuando no exista suficiente informacion para determinar el area correspondiente.


In [38]:
df_carreras_unicas = (
    df_mapeo[["Carrera"]].drop_duplicates().sort_values("Carrera").reset_index(drop=True)
)
print("Total de carreras: ",len(df_carreras_unicas))

display(df_carreras_unicas.head(20))

Total de carreras:  264


,Carrera
0,Abogacía
1,Abogacía - Modalidad Online
2,Actuario
3,Contador Publico
4,Enfermeria
5,Enfermería
6,Enfermería Universitaria
7,Ingeniería Agronómica
8,Lic en Fonoaudiología
9,Lic en Inteligencía de Negocios Digitales


## Clasificacion mediante palabras clave.
### Se utilizaran palabras clave presentes en los nombres de las carreras para generar una primera propuesta de clasificacion.
### Esta asignacion sera preliminar y debera validarse posteriormente, ya que una misma palabra puede aparecer en carreras pertenecientes a diferentes campos academicsos

In [39]:
reglas_areas = {
    "AC01":[
        "matematica","fisica","quimica","biologia","geologia","estadistica","hidrocarburos","geociencias"
    ],
    "AC02":[
        "enfermeria","medicina","nutricion","odontologia","laboratorio clinico","fonoaudiologia","terapia ocupacional"
    ],
    "AC03":[
        "informatica","programacion","sistemas","computacion","software","redes","inteligencia artificial","robotica","ciencia de datos"
    ],
    "AC04":[
        "historia","sociologia","psicologia","trabajo social","comunicacion","abogacia","ciencia politica","criminologia","relaciones internacionales"
    ],
    "AC05":[
        "profesorado","docente","educacion"
    ],
    "AC06":[
        "administracion","contabilidad","economia","marketing","actuario","comercio","martillero","negocios"
    ],
    "AC07":[
        "diseño","artes","musica","teatro","fotografia","periodismo","publicidad"
    ],
    "AC08":[
        "agronomia","agropecuaria","ambiente","ambiental","forestal","produccion animal"
    ]
}
print("dicionario de reglas creado correctamente")

dicionario de reglas creado correctamente


### Crear dos funciones para la clasificacion preliminar

In [40]:
def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize("NFD", texto)

    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )


def clasificar_carrera(carrera, categoria):

    categoria = normalizar_texto(categoria)

    for nombre_categoria, codigo in mapeo_directo.items():
        if normalizar_texto(nombre_categoria) == categoria:
            return codigo

    nombre = normalizar_texto(carrera)
    coincidencias = []

    for codigo, palabras in reglas_areas.items():
        if any(
            normalizar_texto(palabra) in nombre
            for palabra in palabras
        ):
            coincidencias.append(codigo)

    if len(coincidencias) == 1:
        return coincidencias[0]

    return "REVISAR"

### Aplicar reglas

In [41]:
df_mapeo["Codigo_area"] = df_mapeo.apply(
    lambda fila: clasificar_carrera(
        fila["Carrera"],
        fila["Categoria"]
    ),
    axis=1
)

df_mapeo["Area_principal"] = df_mapeo["Codigo_area"].map(
    areas_conocimiento
)

df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR",
    "Area_principal"
] = "Pendiente de revision"

display(df_mapeo.head(20))

,Carrera,Categoria,Codigo_area,Area_principal,Subarea,Criterio_asignacion
0,Abogacía,Derecho,AC04,Ciencias Sociales y Humanas,,
1,Abogacía - Modalidad Online,Ciencias Sociales,AC04,Ciencias Sociales y Humanas,,
2,Actuario,Negocio Y Administracion,AC06,"Economia, Administracion y Negocios",,
3,Contador Publico,Negocio Y Administracion,AC06,"Economia, Administracion y Negocios",,
4,Enfermeria,Tecnicaturas,AC02,Salud y Ciencias de la Vida,,
5,Enfermería,Tecnicaturas,AC02,Salud y Ciencias de la Vida,,
6,Enfermería,Salud Y Psicologia,AC02,Salud y Ciencias de la Vida,,
7,Enfermería Universitaria,Salud Y Psicologia,AC02,Salud y Ciencias de la Vida,,
8,Ingeniería Agronómica,Rural,AC08,"Ambiente, Produccion y Ciencias Agropecuarias",,
9,Lic en Fonoaudiología,Salud Y Psicologia,AC02,Salud y Ciencias de la Vida,,


### Contar los resultados

In [42]:
print("Distribucion de la clasificacion: ")
display(df_mapeo["Area_principal"].value_counts(dropna=False))

Distribucion de la clasificacion: 


Area_principal
Pendiente de revision                            104
Economia, Administracion y Negocios               46
Ciencias Sociales y Humanas                       40
Ciencias Exactas y Naturales                      30
Arte, Diseño y Comunicacion                       24
Educacion y Formacion Docente                     23
Tecnologia e Informatica                          20
Ambiente, Produccion y Ciencias Agropecuarias     17
Salud y Ciencias de la Vida                        8
Name: count, dtype: int64

### Identificar pendientes

In [43]:
print("Carreras pendientes de revision: ")

print((df_mapeo["Codigo_area"] == "REVISAR").sum())

print("\nCarreras clasificadas: ")

print((df_mapeo["Codigo_area"] != "REVISAR").sum())

Carreras pendientes de revision: 
104

Carreras clasificadas: 
208


In [44]:
df_pendientes = (
    df_mapeo[df_mapeo["Codigo_area"] == "REVISAR"][["Carrera","Categoria"]]
        .drop_duplicates().sort_values(["Categoria","Carrera"])
)
print("Total de combinaciones pendientes: ",len(df_pendientes))

display(df_pendientes.head(30))

Total de combinaciones pendientes:  104


,Carrera,Categoria
128,Profesorado de Educación Física,Deportes
129,Profesorado de Educación Física - Orientación ...,Deportes
250,Tecnicatura superior en entrenamiento deportivo,Deportes
46,Licenciatura en Higiene y Seguridad,Medio Ambiente Y Sustentabilidad
260,Tecnicatura superior en gestión de energías re...,Medio Ambiente Y Sustentabilidad
265,Tecnicatura superior en gestión de las energía...,Medio Ambiente Y Sustentabilidad
59,Prof. Educ. Sec. en Ciencias Políticas,Profesorados
65,Prof. de Educ Especial con orientación en Cieg...,Profesorados
66,Prof. de Educ Especial en Discapacidad Neuromo...,Profesorados
72,Prof. de Educ. Esp. con orientación en Sordos ...,Profesorados


### Evitar clasificar carreras repetidas

In [46]:
df_pendientes_unicos = (
    df_pendientes[["Carrera"]].drop_duplicates().sort_values("Carrera").reset_index(drop=True)
)

print("Combinacioones pendientes: ",len(df_pendientes))

print("Carreras pendientes unicas: ",len(df_pendientes_unicos))

display(df_pendientes_unicos)

Combinacioones pendientes:  104
Carreras pendientes unicas:  87


,Carrera
0,Licenciatura en Bromatología
1,Licenciatura en Higiene y Seguridad
2,Prof. Educ. Sec. en Ciencias Políticas
3,Prof. de Educ Especial con orientación en Cieg...
4,Prof. de Educ Especial en Discapacidad Neuromo...
...,...
82,Tecnicatura superior en recursos humanos
83,Tecnicatura superior en soporte de infraestruc...
84,Tecnicatura superior en tecnología de los alim...
85,Tecnicatura superior en traducción técnico cie...


### Revisar las 87 carreras pendientes con nombre completo

In [47]:
with pd.option_context("display.max_rows", None):
    display(df_pendientes_unicos)

,Carrera
0,Licenciatura en Bromatología
1,Licenciatura en Higiene y Seguridad
2,Prof. Educ. Sec. en Ciencias Políticas
3,Prof. de Educ Especial con orientación en Cieg...
4,Prof. de Educ Especial en Discapacidad Neuromo...
5,Prof. de Educ. Esp. con orientación en Sordos ...
6,Prof. de Educ. Inicial
7,Prof. de Educ. Primaria con orientación en EPJA
8,Prof. de Educ. Primaria con orientación en Edu...
9,Prof. de Educ. Primaria con orientación en Int...


### Crear funcion para la revision de carreras pendientes


In [48]:
def clasificar_pendiente(carrera, categoria):
    nombre = normalizar_texto(carrera)
    categoria = normalizar_texto(categoria)

    # Clasificación directa según la categoría original
    if categoria == "profesorados":
        return "AC05"

    if categoria in ["salud y psicologia", "salud y psicología"]:
        return "AC02"

    if categoria == "idiomas":
        return "AC05"

    if categoria == "deportes":
        return "AC05" if "profesorado" in nombre else "AC02"

    # Clasificación por especialidad
    reglas = {
        "AC02": [
            "bromatologia", "higiene y seguridad", "enfermeria",
            "hemoterapia", "bioimagenes", "agente sanitario",
            "acompañamiento terapeutico", "laboratorio de analisis",
            "laboratorio clinico", "farmacia", "esterilizacion",
            "podologia", "protesis dental", "psicomotricidad",
            "instrumentacion quirurgica", "servicios de salud",
            "psicologia"
        ],

        "AC03": [
            "informatica", "software", "ciencia de datos",
            "inteligencia artificial", "robotica", "mecatronica",
            "mecanica", "mantenimiento industrial", "automatizacion",
            "sistemas electricos", "infraestructura tecnologica"
        ],

        "AC04": [
            "derechos humanos", "gestion juridica", "juridica",
            "ciencia politica", "historia", "geografia",
            "sociologia", "filosofia", "trabajo social",
            "ninez", "adolescencia y familia", "mediacion",
            "criminologia", "criminalistica"
        ],

        "AC06": [
            "administracion", "gestion empresarial", "logistica",
            "comercializacion", "marketing", "contable",
            "financiera", "finanzas", "gastronomica",
            "turismo", "hoteleria", "cocina", "ventas",
            "cooperativa", "recursos humanos", "empresas"
        ],

        "AC07": [
            "arte", "diseno", "musica", "danza", "teatro",
            "pintura", "ceramica", "comunicacion social",
            "medios digitales"
        ],

        "AC08": [
            "ambiental", "medio ambiente", "energias renovables",
            "agropecuaria", "agronomica", "produccion agricola",
            "produccion animal", "produccion rural",
            "desarrollo rural", "quimica minera",
            "quimica agroindustrial", "mecanizacion agricola",
            "tecnologia de los alimentos"
        ],

        "AC01": [
            "matematica", "fisica", "quimica", "biologia",
            "ciencias biologicas", "geociencias", "estadistica"
        ]
    }

    # Priorizar áreas con palabras específicas
    for codigo, palabras in reglas.items():
        if any(palabra in nombre for palabra in palabras):
            return codigo

    # Categorías que permiten una asignación general
    if categoria == "medio ambiente y sustentabilidad":
        return "AC08"

    if categoria == "turismo hoteleria y cocina":
        return "AC06"

    return "REVISAR"


# Aplicar únicamente a las carreras que siguen pendientes
mascara_pendientes = df_mapeo["Codigo_area"] == "REVISAR"

df_mapeo.loc[mascara_pendientes, "Codigo_area"] = df_mapeo.loc[
    mascara_pendientes
].apply(
    lambda fila: clasificar_pendiente(
        fila["Carrera"],
        fila["Categoria"]
    ),
    axis=1
)

# Actualizar los nombres de las áreas
df_mapeo["Area_principal"] = df_mapeo["Codigo_area"].map(
    areas_conocimiento
)

df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR",
    "Area_principal"
] = "Pendiente de revision"

# Registrar el criterio de esta modificación
df_mapeo.loc[
    df_mapeo["Codigo_area"] != "REVISAR",
    "Criterio_asignacion"
] = "Clasificacion adicional por nombre y categoria"

# Mostrar resultados
print("Distribución actualizada:")
display(df_mapeo["Area_principal"].value_counts())

print("Carreras que todavía necesitan revisión:")
df_pendientes_finales = df_mapeo[
    df_mapeo["Codigo_area"] == "REVISAR"
][["Carrera", "Categoria"]].drop_duplicates()

display(df_pendientes_finales)

print("Total de combinaciones clasificadas:",
      (df_mapeo["Codigo_area"] != "REVISAR").sum())

print("Total de combinaciones pendientes:",
      len(df_pendientes_finales))


Distribución actualizada:


Area_principal
Economia, Administracion y Negocios              66
Educacion y Formacion Docente                    54
Ciencias Sociales y Humanas                      45
Salud y Ciencias de la Vida                      35
Ciencias Exactas y Naturales                     30
Arte, Diseño y Comunicacion                      25
Ambiente, Produccion y Ciencias Agropecuarias    24
Tecnologia e Informatica                         24
Pendiente de revision                             9
Name: count, dtype: int64

Carreras que todavía necesitan revisión:


,Carrera,Categoria
187,Tecnicatura Superior en Acompañamiento Terapéu...,Tecnicaturas
196,Tecnicatura Superior en Gestión y Prevención d...,Tecnicaturas
206,Tecnicatura Universitaria en Producción de Ani...,Tecnicaturas
216,Tecnicatura superior en acompañamiento terapéu...,Tecnicaturas
218,Tecnicatura superior en acompañante terapéutico,Tecnicaturas
251,Tecnicatura superior en entrenamiento deportivo,Tecnicaturas
279,Tecnicatura superior en indumentaria,Tecnicaturas
304,Tecnicatura superior en soporte de infraestruc...,Tecnicaturas
309,Tecnicatura superior en traducción técnico cie...,Tecnicaturas


Total de combinaciones clasificadas: 303
Total de combinaciones pendientes: 9


### Revisar las 9 pendientes

In [49]:
display(
    df_pendientes_finales.sort_values(
        by=["Categoria", "Carrera"]
    ).reset_index(drop=True)
)

,Carrera,Categoria
0,Tecnicatura Superior en Acompañamiento Terapéu...,Tecnicaturas
1,Tecnicatura Superior en Gestión y Prevención d...,Tecnicaturas
2,Tecnicatura Universitaria en Producción de Ani...,Tecnicaturas
3,Tecnicatura superior en acompañamiento terapéu...,Tecnicaturas
4,Tecnicatura superior en acompañante terapéutico,Tecnicaturas
5,Tecnicatura superior en entrenamiento deportivo,Tecnicaturas
6,Tecnicatura superior en indumentaria,Tecnicaturas
7,Tecnicatura superior en soporte de infraestruc...,Tecnicaturas
8,Tecnicatura superior en traducción técnico cie...,Tecnicaturas


### Agrupar las 9 carreras pendientes

In [50]:
df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR",
    "Codigo_area"
] = df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR"
].apply(
    lambda fila: (
        "AC02" if "acompanamiento terapeutico" in normalizar_texto(fila["Carrera"])
        else "AC04" if "gestion y prevencion" in normalizar_texto(fila["Carrera"])
        else "AC08" if "produccion de animales" in normalizar_texto(fila["Carrera"])
        else "AC02" if "entrenamiento deportivo" in normalizar_texto(fila["Carrera"])
        else "AC07" if "indumentaria" in normalizar_texto(fila["Carrera"])
        else "AC03" if "soporte de infraestructura" in normalizar_texto(fila["Carrera"])
        else "AC04" if "traduccion tecnico cientifica" in normalizar_texto(fila["Carrera"])
        else "REVISAR"
    ),
    axis=1
)

# Actualizar el nombre del área correspondiente
df_mapeo["Area_principal"] = df_mapeo["Codigo_area"].map(areas_conocimiento)

df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR",
    "Area_principal"
] = "Pendiente de revision"

# Actualizar el criterio de asignación
df_mapeo.loc[
    df_mapeo["Criterio_asignacion"] == "REVISAR",
    "Criterio_asignacion"
] = "Clasificacion especifica de carreras pendientes"

# Mostrar resultados
print("Distribución final de áreas:")
display(df_mapeo["Area_principal"].value_counts())

print("Combinaciones clasificadas:",
      (df_mapeo["Codigo_area"] != "REVISAR").sum())

print("Combinaciones pendientes:",
      (df_mapeo["Codigo_area"] == "REVISAR").sum())


Distribución final de áreas:


Area_principal
Economia, Administracion y Negocios              66
Educacion y Formacion Docente                    54
Ciencias Sociales y Humanas                      47
Salud y Ciencias de la Vida                      38
Ciencias Exactas y Naturales                     30
Arte, Diseño y Comunicacion                      26
Ambiente, Produccion y Ciencias Agropecuarias    25
Tecnologia e Informatica                         25
Pendiente de revision                             1
Name: count, dtype: int64

Combinaciones clasificadas: 311
Combinaciones pendientes: 1


### Buscar la carrera pendiente

In [52]:

display(
    df_mapeo.loc[
        df_mapeo["Codigo_area"] == "REVISAR",
        ["Carrera", "Categoria"]
    ]
)


,Carrera,Categoria
218,Tecnicatura superior en acompañante terapéutico,Tecnicaturas


### Agrupar la carrera pendienta a "**AC02 – Salud y Ciencias de la Vida**"

In [53]:

df_mapeo.loc[
    df_mapeo["Codigo_area"] == "REVISAR",
    "Codigo_area"
] = "AC02"

df_mapeo["Area_principal"] = df_mapeo["Codigo_area"].map(areas_conocimiento)

df_mapeo.loc[
    df_mapeo["Criterio_asignacion"] == "REVISAR",
    "Criterio_asignacion"
] = "Clasificacion especifica de carrera pendiente"

print("Combinaciones clasificadas:",
      (df_mapeo["Codigo_area"] != "REVISAR").sum())

print("Combinaciones pendientes:",
      (df_mapeo["Codigo_area"] == "REVISAR").sum())


Combinaciones clasificadas: 312
Combinaciones pendientes: 0


## Craer un nuevo .csv para usar en la app vocacional

In [55]:
df_mapeo.to_csv(
    "../Datos/Procesados/mapeo_carreras_areas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado correctamente.")

Archivo guardado correctamente.
